# GEO · E2b — the operator redesign (relations as rotations, second attempt)

**Why:** E2's learned rotations collapsed to near-identity (1.8°–7.3° — symmetric both-direction training with 1-vs-all dot-product scoring let *entity placement* absorb all the loss), so the relations-as-rotations hypothesis was never fairly tested. The vetting window then confirmed the consequence in the field: the synonym and opposition mining heads surface related pairs but cannot tell types apart. E2b fixes the configuration and re-asks the questions **on the vetted graph** (the 59-edge session-117 delta is applied in-notebook — the public repo clone alone is 5,185; after the delta it matches the working dictionary at 5,240).

**Variants:**
- **B0 (control):** E2's exact configuration (dot-product 1-vs-all CE, random init) on the *vetted* graph — isolates what graph growth alone changes.
- **B1 (the fix):** distance scoring ‖h⊗r̂ − t‖ with self-adversarial negative sampling (RotatE-style) + relations initialized near **180°** (the honest non-identity stable point for symmetric relations) + an identity-repelling regularizer on the rotation scalar.

**Pre-registered questions** (per variant): **Q1** LINK-LEARNABLE — filtered MRR ≥ 0.15 AND Hits@10 ≥ 0.30 · **Q2** OPERATOR-ORDERING — rotation means synonym < affinity < complement AND a non-collapse floor: complement mean rotation ≥ 30° · **Q3** TOPOLOGY→METRIC — |Spearman| ≥ 0.30 vs the grounded angles. *Note: splits differ from E2 (the graph grew), so B0-vs-B1 on identical splits is the clean comparison; E2's historical numbers are context only.*

**How to run:** Runtime ▸ Change runtime type ▸ **T4 GPU** ▸ Save · Runtime ▸ **Run all**. ≈20–30 min. Output: `geo_e2b_results.zip` (browser download at the end; optional Drive cell last).

*Staged 2026-08-20 · follows E2 (pass doc §12) + the vetting return (§13).*

In [ ]:
# ── Setup: GPU check, repo clone ──────────────────────────────────────────────
import subprocess
from pathlib import Path
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'NONE DETECTED')
import torch
assert torch.cuda.is_available(), (
    'No GPU. Colab menu: Runtime > Change runtime type > Hardware accelerator: T4 GPU, then Run all again.')
REPO = Path('/content/agi-semantic-core')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1',
        'https://github.com/QAv2/agi-semantic-core.git', str(REPO)], check=True)
DB = REPO/'db'/'semantic.db'
assert DB.exists()
RESULTS = Path('/content/geo2b_results'); RESULTS.mkdir(exist_ok=True)
SEED = 42

In [ ]:
# ── Apply the vetting delta (session 117: 55 inserts + 4 retypes) ─────────────
import sqlite3, json
DELTA = json.loads(r'''[["SHADOW", "THIS", "opposition", 152.1, 91.74], ["WITNESS", "AWARENESS", "affinity", 30.29, 26.05], ["OPPRESSION", "DELIVERANCE", "affinity", 30.81, 25.34], ["LIMITATION", "FREEDOM", "complement", 90.43, 49.46], ["TRIUMPH", "FAIL", "complement", 94.7294, 57.098], ["SADNESS", "DELIGHT", "complement", 86.3343, 58.3836], ["DISSOLVE", "EMERGE", "complement", 79.6851, 53.0669], ["JOY", "SADNESS", "complement", 83.4214, 58.7208], ["ANXIETY", "WILL", "complement", 80.168, 55.8689], ["ABOVE", "DIRECTION", "affinity", 17.6213, 28.2139], ["SORROW", "EUPHORIA", "complement", 84.4916, 57.9192], ["SEEK", "REFUSE", "opposition", 128.4588, 76.2666], ["SEE", "FAITH", "complement", 100.5265, 60.8176], ["SUCCEED", "LOSS", "complement", 95.087, 55.0185], ["ABUNDANCE", "DEFICIT", "complement", 91.9104, 57.2176], ["DISMAY", "BLISS", "complement", 94.9483, 59.8552], ["SADNESS", "BLISS", "complement", 92.0427, 61.3466], ["DISMAY", "EUPHORIA", "complement", 90.7385, 57.5786], ["ANXIETY", "ASSURANCE", "complement", 78.9799, 52.7973], ["SPARK", "EVAPORATE", "complement", 94.1674, 61.6372], ["CALM", "SHOCK", "complement", 97.9223, 65.8253], ["BIG", "MINUTE_SIZE", "complement", 91.0292, 53.8427], ["STILLNESS", "ACTIVE", "complement", 106.7249, 73.8055], ["HAPPINESS", "DISMAY", "complement", 94.5072, 62.2769], ["SUFFER", "WITNESS", "complement", 109.9307, 65.3879], ["COAX", "DEMAND", "complement", 90.046, 53.9528], ["GRIEF", "ELATION", "complement", 79.0314, 52.7862], ["JOY", "GRIEF", "complement", 71.5088, 50.8117], ["FREEDOM", "WITNESS", "affinity", 36.4482, 33.1168], ["HONOR", "ADMIRATION", "synonym", 17.0539, 16.7026], ["LOVE", "ACCEPT", "affinity", 6.3024, 15.8569], ["BODY", "CHARACTER", "complement", 93.529, 39.7637], ["BECOMES", "TRANSFORMATION", "synonym", 20.3071, 24.4963], ["ABSORB", "PERMEATE", "affinity", 19.6212, 26.9665], ["INCEPTION", "SOLSTICE", "affinity", 29.933, 28.7513], ["HAPPINESS", "FEELING", "affinity", 15.0212, 21.3366], ["LEAD", "COMPLY", "complement", 87.9594, 49.0394], ["FAST", "HURRY", "synonym", 4.3707, 21.825], ["TRUTH", "FRAGMENTATION", "opposition", 149.6601, 83.5605], ["SOLID", "STEADY", "synonym", 24.2669, 32.0481], ["GENESIS", "REVIVAL", "synonym", 0.0659, 11.992], ["GRATITUDE", "FEELING", "affinity", 13.9295, 18.2849], ["ABSORB", "GLEAM", "complement", 101.8667, 53.48], ["OBSCURE", "RADIANT", "complement", 118.7226, 67.0178], ["STRIKE", "HURRY", "affinity", 12.6892, 9.6347], ["OBSCURE", "EMERGE", "complement", 102.5318, 64.6851], ["SLOW", "ERODE", "affinity", 13.1806, 20.0889], ["BECOME", "EMERGE", "affinity", 29.6434, 22.4139], ["WITHER", "DECAY", "synonym", 14.5322, 10.9237], ["ALERT", "DORMANT", "opposition", 168.9854, 89.3433], ["SURGE", "STAGNATION", "opposition", 149.8245, 80.9184], ["WILL", "ACTION", "affinity", 15.4749, 14.4853], ["DARK", "DAWN", "complement", 82.1229, 62.0463], ["DESPAIR", "FAITH", "complement", 81.3821, 46.4708], ["MEEK", "ACCOMMODATE", "affinity", 23.8744, 38.1941], ["CLARITY", "DECIDE", "affinity", 56.4848, 34.5983], ["THOUGHT", "COMPREHEND", "affinity", 8.1318, 22.6052], ["BACKWARD", "BEFORE", "affinity", 11.8979, 42.2762], ["PEACE", "WAR", "complement", 107.8056, 61.6889]]''')
con = sqlite3.connect(DB)
cur = con.cursor()
name_id = {n.upper(): i for n, i in cur.execute('SELECT name, id FROM concepts')}
ins = upd = 0
for n1, n2, rt, a4, a8 in DELTA:
    i1, i2 = name_id[n1.upper()], name_id[n2.upper()]
    cur.execute("""UPDATE relations SET rel_type=?, angle_4d=?, angle_8d=?, session=117
                   WHERE (concept1_id=? AND concept2_id=?) OR (concept1_id=? AND concept2_id=?)""",
                (rt, a4, a8, i1, i2, i2, i1))
    if cur.rowcount == 0:
        cur.execute("""INSERT INTO relations (concept1_id, concept2_id, rel_type, angle_4d, angle_8d, session)
                       VALUES (?,?,?,?,?,117)""", (i1, i2, rt, a4, a8))
        ins += 1
    else:
        upd += 1
con.commit()
n = cur.execute('SELECT COUNT(*) FROM relations').fetchone()[0]
print(f'delta applied: {ins} inserted, {upd} updated -> {n} relations (expected 5240)')
assert n == 5240, 'delta application mismatch'
con.close()

In [ ]:
# ── Dictionary extraction (same code/seed; splits reflect the grown graph) ────
import numpy as np, sqlite3
from collections import Counter
rng = np.random.default_rng(SEED)
con = sqlite3.connect(DB); con.row_factory = sqlite3.Row
DIMS = ['x','y','z','e','f','g','h','fx','fy','fz','fe','ff','fg','fh']
rows = con.execute(f"SELECT id,name,description,{','.join(DIMS)} FROM concepts ORDER BY id").fetchall()
names = [r['name'] for r in rows]
Y14 = np.array([[r[d] for d in DIMS] for r in rows], dtype=np.float64)
Y7 = Y14[:, :7]
id2idx = {r['id']: i for i, r in enumerate(rows)}
pairs = []
for r in con.execute("SELECT concept1_id c1, concept2_id c2, rel_type, angle_4d, angle_8d FROM relations"):
    if r['c1'] not in id2idx or r['c2'] not in id2idx: continue
    ta = r['angle_8d'] if r['angle_8d'] else r['angle_4d']
    if not ta: continue
    pairs.append((id2idx[r['c1']], id2idx[r['c2']], r['rel_type'], float(ta)))
print(f'{len(pairs)} usable relations —', dict(Counter(p[2] for p in pairs)))
by_type = {}
for p in pairs: by_type.setdefault(p[2], []).append(p)
train_rel, test_rel = [], []
for t, ps in sorted(by_type.items()):
    idx = rng.permutation(len(ps)); cut = int(0.8*len(ps))
    train_rel += [ps[i] for i in idx[:cut]]; test_rel += [ps[i] for i in idx[cut:]]
print(f'split: {len(train_rel)} train / {len(test_rel)} held out')
def angle7(i, j):
    a, b = Y7[i], Y7[j]
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-9 or nb < 1e-9: return None
    return float(np.degrees(np.arccos(np.clip(a@b/(na*nb), -1, 1))))
related = {(min(p[0],p[1]), max(p[0],p[1])) for p in pairs}
rand_pairs, seen = [], set()
while len(rand_pairs) < 12000:
    i, j = (int(v) for v in rng.integers(0, len(names), 2))
    key = (min(i,j), max(i,j))
    if i == j or key in related or key in seen: continue
    a = angle7(i, j)
    if a is None: continue
    seen.add(key); rand_pairs.append((i, j, 'random', a))
rand_test = rand_pairs[10000:]

In [ ]:
# ── Shared machinery: model, training modes, evals ────────────────────────────
import torch, torch.nn as nn, numpy as np, time
from collections import defaultdict
from scipy.stats import spearmanr
dev = 'cuda'
REL_TYPES = sorted({p[2] for p in pairs})
r2i = {r: i for i, r in enumerate(REL_TYPES)}
N, M, K = len(names), len(REL_TYPES), 24

def edges_of(plist):
    e = [(p[0], r2i[p[2]], p[1]) for p in plist] + [(p[1], r2i[p[2]], p[0]) for p in plist]
    return torch.tensor(e, dtype=torch.long, device=dev)
train_e, test_e = edges_of(train_rel), edges_of(test_rel)
known_tails = defaultdict(list)
for p in pairs:
    known_tails[(p[0], r2i[p[2]])].append(p[1]); known_tails[(p[1], r2i[p[2]])].append(p[0])

def hamilton(q, p):
    a1,b1,c1,d1 = q.unbind(-1); a2,b2,c2,d2 = p.unbind(-1)
    return torch.stack([a1*a2 - b1*b2 - c1*c2 - d1*d2,
                        a1*b2 + b1*a2 + c1*d2 - d1*c2,
                        a1*c2 - b1*d2 + c1*a2 + d1*b2,
                        a1*d2 + b1*c2 - c1*b2 + d1*a2], -1)

class QuatE(nn.Module):
    def __init__(self, init_half_turn=False):
        super().__init__()
        self.E = nn.Parameter(torch.randn(N, K, 4) * 0.3)
        R = torch.randn(M, K, 4)
        if init_half_turn:
            R[..., 0] = torch.randn(M, K) * 0.05          # tiny scalar -> rotation near 180 deg
        self.R = nn.Parameter(R * 0.5)
    def rhat(self, r_idx):
        r = self.R[r_idx]
        return r / r.norm(dim=-1, keepdim=True).clamp_min(1e-9)
    def rotate(self, h_idx, r_idx):
        return hamilton(self.E[h_idx], self.rhat(r_idx))
    def score_all_dot(self, h_idx, r_idx):
        hr = self.rotate(h_idx, r_idx).reshape(len(h_idx), -1)
        return hr @ self.E.reshape(N, -1).T
    def dist(self, h_idx, r_idx, t_idx):
        hr = self.rotate(h_idx, r_idx)
        return (hr - self.E[t_idx]).reshape(len(h_idx), -1).norm(dim=1)
    def dist_all(self, h_idx, r_idx):
        hr = self.rotate(h_idx, r_idx).reshape(len(h_idx), 1, -1)
        return (hr - self.E.reshape(1, N, -1)).norm(dim=2)

def train_model(mode, epochs=400, gamma=4.0, lam_id=0.5, neg=64):
    torch.manual_seed(SEED); np.random.seed(SEED)
    model = QuatE(init_half_turn=(mode == 'dist')).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)
    ce = nn.CrossEntropyLoss()
    t0 = time.time()
    for ep in range(1, epochs+1):
        perm = torch.randperm(len(train_e), device=dev)
        for s in range(0, len(train_e), 1024):
            b = train_e[perm[s:s+1024]]
            if mode == 'dot':
                loss = ce(model.score_all_dot(b[:,0], b[:,1]), b[:,2])
            else:
                d_pos = model.dist(b[:,0], b[:,1], b[:,2])
                negs = torch.randint(0, N, (len(b), neg), device=dev)
                hr = model.rotate(b[:,0], b[:,1]).reshape(len(b), 1, -1)
                d_neg = (hr - model.E[negs].reshape(len(b), neg, -1)).norm(dim=2)
                w = torch.softmax(gamma - d_neg, dim=1).detach()      # self-adversarial
                loss = (-torch.nn.functional.logsigmoid(gamma - d_pos).mean()
                        - (w * torch.nn.functional.logsigmoid(d_neg - gamma)).sum(1).mean())
                loss = loss + lam_id * model.rhat(torch.arange(M, device=dev))[..., 0].abs().mean()
            opt.zero_grad(); loss.backward(); opt.step()
        if ep % 100 == 0 or ep == 1:
            print(f'  [{mode}] epoch {ep:3d} loss {loss.item():.4f} ({time.time()-t0:.0f}s)')
    return model

def eval_all(model, mode, tag):
    model.eval(); res = {'tag': tag}
    ranks = []
    with torch.no_grad():
        for s in range(0, len(test_e), 512):
            b = test_e[s:s+512]
            sc = model.score_all_dot(b[:,0], b[:,1]) if mode == 'dot' else -model.dist_all(b[:,0], b[:,1])
            for row in range(len(b)):
                h, r, t = (int(v) for v in b[row])
                gold = sc[row, t].item()
                m = sc[row].clone()
                for kt in known_tails[(h, r)]: m[kt] = -1e9
                ranks.append((r, int((m > gold).sum().item()) + 1))
    ks = np.array([k for _, k in ranks], dtype=float)
    res['MRR'] = float((1/ks).mean()); res['Hits@10'] = float((ks <= 10).mean())
    res['per_type_MRR'] = {t: float(np.mean([1/k for r, k in ranks if r == r2i[t]]) if any(r == r2i[t] for r,_ in ranks) else 0)
                           for t in REL_TYPES}
    with torch.no_grad():
        Rn = model.rhat(torch.arange(M, device=dev))
        theta = torch.rad2deg(2*torch.arccos(Rn[..., 0].abs().clamp(0, 1)))
        res['rot_mean'] = {t: float(theta[r2i[t]].mean()) for t in REL_TYPES}
        Ef = model.E.reshape(N, -1)
        Ef = (Ef / Ef.norm(dim=1, keepdim=True)).cpu().numpy()
    tm = test_rel + rand_test
    true_a = np.array([p[3] for p in tm])
    pred_a = np.array([float(np.degrees(np.arccos(np.clip(Ef[p[0]] @ Ef[p[1]], -1, 1)))) for p in tm])
    res['topometric_spearman'] = float(spearmanr(true_a, pred_a)[0])
    model.train()
    return res

In [ ]:
# ── B0: control — E2 configuration on the vetted graph ────────────────────────
m0 = train_model('dot')
res0 = eval_all(m0, 'dot', 'B0_control')
print(res0['MRR'], res0['Hits@10'], res0['topometric_spearman'])
print('rotations:', {k: round(v,1) for k,v in res0['rot_mean'].items()})

In [ ]:
# ── B1: the fix — distance scoring + near-180 init + identity repulsion ───────
m1 = train_model('dist')
res1 = eval_all(m1, 'dist', 'B1_fix')
print(res1['MRR'], res1['Hits@10'], res1['topometric_spearman'])
print('rotations:', {k: round(v,1) for k,v in res1['rot_mean'].items()})

In [ ]:
# ── Pre-registered verdict ────────────────────────────────────────────────────
import json
out = {}
for res in [res0, res1]:
    rot = res['rot_mean']
    q1 = bool(res['MRR'] >= 0.15 and res['Hits@10'] >= 0.30)
    q2 = bool(rot.get('synonym',1e9) < rot.get('affinity',-1) < rot.get('complement',-1)
              and rot.get('complement',0) >= 30)
    q3 = bool(abs(res['topometric_spearman']) >= 0.30)
    out[res['tag']] = {**res, 'Q1': q1, 'Q2': q2, 'Q3': q3}
    print(f"{res['tag']:12s} Q1 {'PASS' if q1 else 'FAIL'} (MRR {res['MRR']:.3f}, H@10 {res['Hits@10']:.3f})  "
          f"Q2 {'PASS' if q2 else 'FAIL'} (syn {rot.get('synonym',0):.0f} aff {rot.get('affinity',0):.0f} "
          f"comp {rot.get('complement',0):.0f} opp {rot.get('opposition',0):.0f})  "
          f"Q3 {'PASS' if q3 else 'FAIL'} (rho {res['topometric_spearman']:.3f})")
json.dump(out, open(RESULTS/'e2b_verdict.json','w'), indent=1)
print()
print('Reading: B0 isolates graph growth; B1 isolates the operator fix. If B1 passes Q2')
print('with meaningful magnitudes, relations-as-rotations was a configuration casualty in')
print('E2, not a dead hypothesis. If B1 also collapses, the hypothesis is in real trouble.')

In [ ]:
# ── Mining sanity: does B1 now know type identity? ────────────────────────────
import numpy as np, torch
existing = {(min(p[0],p[1]), max(p[0],p[1])) for p in pairs}
for rel in ['opposition', 'synonym']:
    ri = torch.tensor([r2i[rel]]*512, device=dev)
    S = np.zeros((N, N), dtype=np.float32)
    with torch.no_grad():
        for s in range(0, N, 512):
            h = torch.arange(s, min(s+512, N), device=dev)
            S[s:s+512] = (-m1.dist_all(h, ri[:len(h)])).cpu().numpy()
    Ssym = (S + S.T) / 2
    iu = np.triu_indices(N, k=1)
    order = np.argsort(-Ssym[iu])
    print(f'\n=== B1 top 10 NEW {rel} proposals (grounded core-and-7D angles attached) ===')
    shown = 0
    for oi in order:
        i, j = int(iu[0][oi]), int(iu[1][oi])
        if (i, j) in existing: continue
        c3 = Y14[i][:3], Y14[j][:3]
        n1_, n2_ = np.linalg.norm(c3[0]), np.linalg.norm(c3[1])
        a3 = float(np.degrees(np.arccos(np.clip(c3[0]@c3[1]/max(n1_*n2_,1e-9), -1, 1))))
        print(f'  {names[i]:016s} {names[j]:016s} score {Ssym[i,j]:7.2f}  core {a3:5.1f}  7d {angle7(i,j):5.1f}')
        shown += 1
        if shown >= 10: break

In [ ]:
# ── Package (+ browser download) ──────────────────────────────────────────────
import shutil, json
zip_path = shutil.make_archive('/content/geo_e2b_results', 'zip', RESULTS)
print('packaged:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print('manual download: Files sidebar ->', zip_path)

In [ ]:
# ── OPTIONAL: copy to Drive ───────────────────────────────────────────────────
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    import shutil; shutil.copy('/content/geo_e2b_results.zip', '/content/drive/MyDrive/geo_e2b_results.zip')
    print('saved to Drive')